A SeqFeature or Location object does not directly store or contain a sequence, it ony stores cordinates (via the location (start, end, strand))which decribes how to get the sequence from the parent sequence. To get the actual sequences, we use the cordinates to extract it from tthe parent sequence. 

for example: consider this gene with the cordinates...

In [9]:
from Bio.Seq import Seq
from Bio.SeqFeature import SeqFeature, FeatureLocation

seq1 = Seq("ACCGAGACGGCAAAGGCTAGCATAGGTATGAGACTTCCTTCCTGCCAGTGCTGAGGAACTGGGAGCCTAC")

feature = SeqFeature(FeatureLocation(5, 18,  strand = -1), type = "gene")

print(feature)



type: gene
location: [5:18](-)
qualifiers:



From The features, we know that the sequence is from position 5-18(0 based indexing), and  it's sequence type is a gene and it's on the reverse strand. We can Extract it from the parent sequence in 2 ways 

1) Manual extraction:

In [10]:
feature_seq = seq1[feature.location.start: feature.location.end].reverse_complement()
print(feature_seq)



AGCCTTTGCCGTC


This slices the parent sequence from index 5 to index 18 then flips and complement it.
This works for simple  features but will fall apart the momen we have compound locations(joins with multiple exons). but we can use an extract() function to handle transsplicing automatically

2). .extract()

In [11]:
feature_seq = feature.extract(seq1)
print(feature_seq)

AGCCTTTGCCGTC


This does everything automatically by slicing the correct region, it will also reverse compliment it if strand is -1 and handle the compound location(joins) by extracting each piece and concatenating them. 

Also notice below how all the length is he same because 18 - 5 is 13, this shows that the output is realy following the location cordinates

In [ ]:
print(len(feature.location))     # 13 - Length of location
print(len(feature))              # 13 length of sequence object itself
print(len(feature_seq))          # 13 length of th extracted sequence

Example of using the extract() function with compound location:
A gene with two exons seperated by an intron, note that only the exons shoukd be spliced into the final mRNA and the intron will be discarded

In [18]:
from Bio.Seq import Seq
from Bio.SeqFeature import SeqFeature, FeatureLocation,CompoundLocation
# a 60bp DNA sequence

seq2 = Seq("AAAAAATGCGTAACGTTTTTTTTTTTTTTTTTTTTTTTTTGGCAAAGGCTAGCAAAAAAA")

print(len(seq2))

# Exon 1: position  6 to 20
exon1 = FeatureLocation(6,20, strand = +1)

# Exon 2: position 40 to 52
exon2 = FeatureLocation(40,52, strand = +1)

seq2pos = CompoundLocation([exon1,exon2])

genefeature = SeqFeature( seq2pos, type = "mRNA")

mrna_Seq = genefeature.extract(seq2)

60


Note that we can also convert a SeqRecord into a string formatted as a special file type by using the format() method. This simply convvers the

In [21]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
# a 60bp DNA sequence

seq3 = SeqRecord(
    Seq("AAAAAATGCGTAACGTTTTTTTTTTTTTTTTTTTTTTTTTGGCAAAGGCTAGCAAAAAAATGACTAGACTAGGTCCCAATTACATACCAGGACACCATTGGCATTCAGATACGGGGTTTGTAC"),
    id = "gi|14150838|gb|AAK54648.1|AF376133_1",
    description="chalcone synthase [Cucumis sativus]",)

print(seq3.format("fasta"))

>gi|14150838|gb|AAK54648.1|AF376133_1 chalcone synthase [Cucumis sativus]
AAAAAATGCGTAACGTTTTTTTTTTTTTTTTTTTTTTTTTGGCAAAGGCTAGCAAAAAAA
TGACTAGACTAGGTCCCAATTACATACCAGGACACCATTGGCATTCAGATACGGGGTTTG
TAC



This converts the data sequence into a Fasta Format

